# Domänundersökning och visualisering

In [1]:
# @title 1. Lägg till domäner du vill undersöka (exempelvis: aftonbladet.se, svt.se, dn.se)
Domän_lista = "" #@param {type:"string"}
DOMAINS = [d.strip() for d in Domän_lista.split(',') if d.strip()]

print(f"Loaded {len(DOMAINS)} domains from form field")
# Note: If you run Cell 4, it will prompt for domains again and overwrite this list.
# You can comment out the input() line in Cell 4 if you want to use these domains.

Loaded 3 domains from form field


In [9]:
# @title 2. Installera och importera biblotek
!pip install python-whois dnspython pyvis networkx matplotlib requests -q

import socket
import requests
import urllib3
import whois
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyvis.network import Network
from IPython.display import IFrame, display as idisplay

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [3]:
import logging
logging.getLogger("whois.whois").setLevel(logging.CRITICAL)
import threading
import random

# @title 3. Kör domänundersökningen

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}

STATUS_COLOR = {
    "ACTIVE":       "#2ecc71",
    "EXPIRED":      "#e74c3c",
    "UNREGISTERED": "#e74c3c",
    "ERROR":        "#95a5a6",
}

# Global lock to serialize WHOIS calls and avoid hammering servers
_whois_lock = threading.Lock()

def parse_date(value):
    if isinstance(value, list):
        value = value[0]
    if isinstance(value, datetime):
        return value, value.strftime("%Y-%m-%d")
    if isinstance(value, str):
        try:
            dt = datetime.strptime(value[:10], "%Y-%m-%d")
            return dt, value[:10]
        except:
            return None, value
    return None, None

def get_whois_info(domain):
    """Fetch WHOIS with throttling + exponential backoff retries."""
    for attempt in range(4):
        try:
            with _whois_lock:
                result = whois.whois(domain)
                time.sleep(random.uniform(1.5, 3.0))   # pause between every call
            return result
        except Exception as e:
            err = str(e)
            if any(x in err for x in ["No match", "Not found", "No entries found", "No whois"]):
                return None
            wait = (2 ** attempt) * 3 + random.uniform(0, 2)   # 3s, 6s, 12s + jitter
            print(f"    [retry {attempt+1}/4] {domain} — waiting {wait:.1f}s ({err[:50]})")
            time.sleep(wait)
    return None   # give up after 4 attempts

def check_domain(domain):
    result = {
        "domain":       domain,
        "ip_address":   None,
        "status":       None,
        "whois_server": None,
        "registrar":    None,
        "created":      None,
        "updated":      None,
        "expires":      None,
        "nameservers":  None,
        "country":      None,
        "http_code":    None,
        "redirects_to": None,
        "error":        None,
    }

    # 1. DNS -> IP
    try:
        result["ip_address"] = socket.gethostbyname(domain)
    except socket.gaierror:
        pass

    # 2. WHOIS
    try:
        d = get_whois_info(domain)

        if d is None:
            result["status"] = "UNREGISTERED"
            return result

        result["whois_server"] = d.whois_server or None
        result["registrar"]    = d.registrar    or None
        result["country"]      = d.country      or None

        ns = d.name_servers
        if isinstance(ns, (list, set)):
            ns = ", ".join(sorted(set(n.lower() for n in ns if n)))
        result["nameservers"] = ns or None

        _, result["created"] = parse_date(d.creation_date)
        _, result["updated"] = parse_date(d.updated_date)
        ed_dt, result["expires"] = parse_date(d.expiration_date)

        if ed_dt:
            result["status"] = "ACTIVE" if ed_dt > datetime.now() else "EXPIRED"
        elif d.domain_name:
            result["status"] = "ACTIVE"
        else:
            result["status"] = "UNREGISTERED"

    except Exception as e:
        result["status"] = "ACTIVE" if result["ip_address"] else "ERROR"
        result["error"]  = str(e)[:80]

    # 3. HTTP check (informational)
    if result["ip_address"]:
        for scheme in ("https", "http"):
            try:
                resp = requests.get(
                    f"{scheme}://{domain}",
                    timeout=8, verify=False,
                    allow_redirects=True,
                    headers=HEADERS,
                )
                result["http_code"] = resp.status_code
                if resp.history:
                    result["redirects_to"] = resp.url
                break
            except Exception:
                continue
        if result["http_code"] is None:
            result["http_code"] = "unreachable"

    return result


# ── Run ───────────────────────────────────────────────────────────────────────
# Using DOMAINS list from the previous cell (Cell 1)

if not DOMAINS:
    print("No domains provided. Please provide at least one domain to proceed in Cell 1.")
    results = [] # Initialize results to avoid NameError later
    df = pd.DataFrame(results) # Create an empty DataFrame
    print("\n[INFO] No domains processed.")
    display(df) # Display the empty DataFrame
else:
    print(f"Checking {len(DOMAINS)} domains (throttled — expect ~{len(DOMAINS)*2/3:.0f}-{len(DOMAINS)*3/3:.0f} mins)..\n")
    results = []

    # 3 workers + lock + sleep = ~1 WHOIS call per 1.5-3s across all threads
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {pool.submit(check_domain, d): d for d in DOMAINS}
        for i, future in enumerate(as_completed(futures), 1):
            res = future.result()
            results.append(res)
            icon = {"ACTIVE": "[OK]", "EXPIRED": "[EXP]", "UNREGISTERED": "[NONE]", "ERROR": "[ERR]"}.get(res["status"], "[?]")
            print(f"[{i:>3}/{len(DOMAINS)}] {icon} {res['domain']:<35} {res['status']:<14} "
                  f"exp:{res['expires'] or 'n/a':<12} http:{res['http_code'] or '-'}")

    df = pd.DataFrame(results).sort_values(["status", "domain"]).reset_index(drop=True)
    print("\n=== SUMMARY ===")
    print(df["status"].value_counts().to_string())
    df.to_csv("domain_report.csv", index=False)
    print("\n[DONE] Saved domain_report.csv")
    display(df)

Checking 3 domains (throttled — expect ~2-3 mins)..

[  1/3] [NONE] usa-aa.com                          UNREGISTERED   exp:n/a          http:-
[  2/3] [OK] doloreshoy.com                      ACTIVE         exp:2027-02-15   http:200
[  3/3] [OK] splinsider.com                      ACTIVE         exp:2027-02-19   http:200

=== SUMMARY ===
status
ACTIVE          2
UNREGISTERED    1

[DONE] Saved domain_report.csv


,domain,ip_address,status,whois_server,registrar,created,updated,expires,nameservers,country,http_code,redirects_to,error
0,doloreshoy.com,76.223.54.146,ACTIVE,grs-whois.hichina.com,Alibaba Cloud Computing Ltd. d/b/a HiChina (ww...,2026-02-15,2026-02-15,2027-02-15,"dns1.hichina.com, dns2.hichina.com, ns1.aftern...",CN,200.0,None,can't compare offset-naive and offset-aware da...
1,splinsider.com,162.0.216.242,ACTIVE,whois.namecheap.com,NAMECHEAP INC,2023-02-19,2026-01-20,2027-02-19,"dns1.registrar-servers.com, dns2.registrar-ser...",IS,200.0,https://www.splinsider.com/,can't compare offset-naive and offset-aware da...
2,usa-aa.com,None,UNREGISTERED,None,None,None,None,None,None,None,NaN,None,None


In [14]:
# @title 4. Avancerad interaktiv SNA

from IPython.display import display as idisplay, HTML, IFrame
import json

idisplay(HTML("""
<style>
    .jp-Cell-outputArea iframe {
        height: 95vh !important;
        min-height: 800px !important;
    }
</style>
"""))

net = Network(height="100%", width="100%", bgcolor="#1a1a2e",
              font_color="white", notebook=True, cdn_resources="in_line")

net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.005,
      "springLength": 140
    },
    "solver": "forceAtlas2Based",
    "stabilization": { "iterations": 150 }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hideEdgesOnDrag": true
  }
}
""")

added_ips = set()
node_data = {}

for row in results:
    domain = row["domain"]
    ip     = row["ip_address"] or "unresolved"
    color  = STATUS_COLOR.get(row["status"], "#95a5a6")
    def f(val): return str(val) if val else "n/a"

    node_data[domain] = {
        "type":         "domain",
        "DOMAIN":       f(row["domain"]),
        "STATUS":       f(row["status"]),
        "IP":           f(row["ip_address"]),
        "HTTP CODE":    f(str(row["http_code"])),
        "REDIRECT":     f(row["redirects_to"]),
        "REGISTRAR":    f(row["registrar"]),
        "WHOIS SERVER": f(row["whois_server"]),
        "COUNTRY":      f(row["country"]),
        "CREATED":      f(row["created"]),
        "UPDATED":      f(row["updated"]),
        "EXPIRES":      f(row["expires"]),
        "NAMESERVERS":  f(row["nameservers"]),
        "ERROR":        f(row["error"]),
        "NOTES":        "",
    }

    net.add_node(domain, label=domain, color=color, shape="dot",
                 size=14, title="Click for details: " + domain,
                 font={"size": 11, "color": "white"})

    if ip not in added_ips:
        shared_domains = [r["domain"] for r in results if (r["ip_address"] or "unresolved") == ip]
        node_data[ip] = {
            "type":           "ip",
            "IP ADDRESS":     ip,
            "DOMAINS HOSTED": str(len(shared_domains)),
            "DOMAIN LIST":    ", ".join(shared_domains),
            "NOTES":          "",
        }
        net.add_node(ip, label=ip,
                     color="#4a90d9" if ip != "unresolved" else "#555",
                     shape="square", size=22,
                     title="Click for details: " + ip,
                     font={"size": 13, "bold": True, "color": "white"})
        added_ips.add(ip)

    net.add_edge(domain, ip, color={"color": color, "opacity": 0.6})

html = net.generate_html()

side_panel_css = """
<style>
body, html { margin: 0; padding: 0; height: 100%; overflow: hidden; }
#mynetwork { width: 100% !important; height: 100vh !important; position: absolute; top: 0; left: 0; }
#side-panel {
    position: fixed; top: 20px; right: 20px; width: 340px; max-height: 90vh;
    background: #1e1e2e; border: 1px solid #4a90d9; border-radius: 8px;
    padding: 16px; overflow-y: auto; z-index: 9999; display: none;
    box-shadow: 0 4px 20px rgba(0,0,0,0.5); font-family: monospace;
}
#side-panel h3 { color: #4a90d9; margin: 0 0 12px 0; font-size: 13px;
    border-bottom: 1px solid #333; padding-bottom: 8px; padding-right: 20px; }
#panel-table { width: 100%; border-collapse: collapse; font-size: 12px; }
#panel-table td { padding: 4px 6px; vertical-align: top; color: #e0e0e0;
    border-bottom: 1px solid #2a2a3e; user-select: text; -webkit-user-select: text; }
#panel-table td:first-child { color: #4a90d9; white-space: nowrap;
    padding-right: 10px; font-weight: bold; width: 110px; }
#panel-table td:last-child { word-break: break-all; }
.edit-input { background: #2a2a3e; border: 1px solid #4a90d9; color: #e0e0e0;
    font-family: monospace; font-size: 12px; width: 100%; padding: 2px 4px;
    border-radius: 3px; box-sizing: border-box; }
.edit-input:focus { outline: none; border-color: #2ecc71; }
.status-active   { color: #2ecc71; font-weight: bold; }
.status-expired  { color: #e74c3c; font-weight: bold; }
.status-unreg    { color: #e74c3c; font-weight: bold; }
.status-error    { color: #95a5a6; font-weight: bold; }
.btn-row { display: flex; gap: 6px; margin-top: 10px; }
.btn { flex: 1; padding: 7px; border: none; border-radius: 4px; cursor: pointer;
    font-family: monospace; font-size: 11px; font-weight: bold; }
#copy-btn { background: #4a90d9; color: white; }
#copy-btn:hover { background: #357abd; }
#save-btn { background: #2ecc71; color: #1a1a2e; }
#save-btn:hover { background: #27ae60; }
#export-btn { width: 100%; margin-top: 6px; padding: 7px; background: #8e44ad;
    color: white; border: none; border-radius: 4px; cursor: pointer;
    font-family: monospace; font-size: 11px; font-weight: bold; }
#export-btn:hover { background: #732d91; }
#copy-confirm { color: #2ecc71; font-size: 11px; text-align: center; margin-top: 6px; display: none; }
#close-btn { position: absolute; top: 10px; right: 12px; background: none;
    border: none; color: #888; cursor: pointer; font-size: 16px; }
#close-btn:hover { color: white; }
#edit-toggle { font-size: 10px; color: #aaa; cursor: pointer; text-decoration: underline;
    display: block; text-align: right; margin-bottom: 6px; }
#edit-toggle:hover { color: white; }
#legend { position: fixed; bottom: 20px; left: 20px; background: #1e1e2e;
    border: 1px solid #333; border-radius: 8px; padding: 10px 14px; z-index: 9999;
    font-family: monospace; font-size: 12px; color: #e0e0e0; }
#legend div { margin: 4px 0; display: flex; align-items: center; gap: 8px; }
.dot { width: 12px; height: 12px; border-radius: 50%; display: inline-block; flex-shrink: 0; }
.square { width: 12px; height: 12px; display: inline-block; flex-shrink: 0; }
</style>
"""

side_panel_html = """
<div id="side-panel">
    <button id="close-btn" onclick="document.getElementById('side-panel').style.display='none'">x</button>
    <h3 id="panel-title">Node Info</h3>
    <span id="edit-toggle" onclick="toggleEdit()">enable editing</span>
    <table id="panel-table"></table>
    <div class="btn-row">
        <button class="btn" id="copy-btn" onclick="copyInfo()">Copy to clipboard</button>
        <button class="btn" id="save-btn" onclick="saveEdits()" style="display:none">Save edits</button>
    </div>
    <button id="export-btn" onclick="exportCSV()">Export all data as CSV</button>
    <div id="copy-confirm"></div>
</div>
<div id="legend">
    <div><span class="dot" style="background:#2ecc71"></span> Active</div>
    <div><span class="dot" style="background:#e74c3c"></span> Expired / Unregistered</div>
    <div><span class="dot" style="background:#95a5a6"></span> Error / Unknown</div>
    <div><span class="square" style="background:#4a90d9"></span> IP Address</div>
</div>
"""

# Bygg JS separat från node_data för att undvika f-sträng + json-kollision
node_data_json = json.dumps(node_data)

side_panel_js = (
    "<script>\n"
    "var nodeData = " + node_data_json + ";\n"
    "var currentNode = null;\n"
    "var editMode = false;\n"
    "var readOnly = new Set(['type', 'DOMAIN', 'IP ADDRESS']);\n"
    "\n"
    "function statusClass(val) {\n"
    "    if (!val) return '';\n"
    "    val = val.toUpperCase();\n"
    "    if (val === 'ACTIVE') return 'status-active';\n"
    "    if (val === 'EXPIRED') return 'status-expired';\n"
    "    if (val === 'UNREGISTERED') return 'status-unreg';\n"
    "    if (val === 'ERROR') return 'status-error';\n"
    "    return '';\n"
    "}\n"
    "\n"
    "function showPanel(nodeId) {\n"
    "    currentNode = nodeId;\n"
    "    var data = nodeData[nodeId];\n"
    "    if (!data) return;\n"
    "    document.getElementById('panel-title').innerText =\n"
    "        data.type === 'ip' ? 'IP: ' + nodeId : 'DOMAIN: ' + nodeId;\n"
    "    renderTable(data);\n"
    "    document.getElementById('side-panel').style.display = 'block';\n"
    "    document.getElementById('copy-confirm').innerText = '';\n"
    "    document.getElementById('copy-confirm').style.display = 'none';\n"
    "}\n"
    "\n"
    "function renderTable(data) {\n"
    "    var table = document.getElementById('panel-table');\n"
    "    table.innerHTML = '';\n"
    "    for (var key in data) {\n"
    "        if (key === 'type') continue;\n"
    "        var row = table.insertRow();\n"
    "        var c1 = row.insertCell(0);\n"
    "        var c2 = row.insertCell(1);\n"
    "        c1.innerText = key;\n"
    "        var val = data[key];\n"
    "        var sc  = statusClass(val);\n"
    "        if (editMode && !readOnly.has(key)) {\n"
    "            var inp = document.createElement('input');\n"
    "            inp.className = 'edit-input';\n"
    "            inp.value = val;\n"
    "            inp.dataset.key = key;\n"
    "            c2.appendChild(inp);\n"
    "        } else {\n"
    "            if (sc) c2.innerHTML = '<span class=\"' + sc + '\">' + val + '</span>';\n"
    "            else c2.innerText = val;\n"
    "        }\n"
    "    }\n"
    "}\n"
    "\n"
    "function toggleEdit() {\n"
    "    editMode = !editMode;\n"
    "    document.getElementById('edit-toggle').innerText = editMode ? 'disable editing' : 'enable editing';\n"
    "    document.getElementById('save-btn').style.display = editMode ? 'inline-block' : 'none';\n"
    "    if (currentNode) renderTable(nodeData[currentNode]);\n"
    "}\n"
    "\n"
    "function saveEdits() {\n"
    "    if (!currentNode) return;\n"
    "    var inputs = document.querySelectorAll('.edit-input');\n"
    "    inputs.forEach(function(inp) {\n"
    "        nodeData[currentNode][inp.dataset.key] = inp.value;\n"
    "    });\n"
    "    editMode = false;\n"
    "    document.getElementById('edit-toggle').innerText = 'enable editing';\n"
    "    document.getElementById('save-btn').style.display = 'none';\n"
    "    renderTable(nodeData[currentNode]);\n"
    "    document.getElementById('copy-confirm').innerText = 'Changes saved!';\n"
    "    document.getElementById('copy-confirm').style.display = 'block';\n"
    "    setTimeout(function() {\n"
    "        document.getElementById('copy-confirm').style.display = 'none';\n"
    "    }, 2000);\n"
    "}\n"
    "\n"
    "function copyInfo() {\n"
    "    if (!currentNode) return;\n"
    "    var data = nodeData[currentNode];\n"
    "    var text = '';\n"
    "    for (var key in data) {\n"
    "        if (key === 'type') continue;\n"
    "        text += key + ': ' + data[key] + '\\n';\n"
    "    }\n"
    "    navigator.clipboard.writeText(text).then(function() {\n"
    "        document.getElementById('copy-confirm').innerText = 'Copied to clipboard!';\n"
    "        document.getElementById('copy-confirm').style.display = 'block';\n"
    "        setTimeout(function() {\n"
    "            document.getElementById('copy-confirm').style.display = 'none';\n"
    "        }, 2000);\n"
    "    });\n"
    "}\n"
    "\n"
    "function exportCSV() {\n"
    "    var allKeys = [];\n"
    "    for (var id in nodeData) {\n"
    "        if (nodeData[id].type === 'domain') {\n"
    "            allKeys = Object.keys(nodeData[id]).filter(k => k !== 'type');\n"
    "            break;\n"
    "        }\n"
    "    }\n"
    "    var rows = [allKeys.join(',')];\n"
    "    for (var id in nodeData) {\n"
    "        if (nodeData[id].type !== 'domain') continue;\n"
    "        var row = allKeys.map(function(k) {\n"
    "            var val = (nodeData[id][k] || '').replace(/\"/g, '\"\"');\n"
    "            return '\"' + val + '\"';\n"
    "        });\n"
    "        rows.push(row.join(','));\n"
    "    }\n"
    "    var blob = new Blob([rows.join('\\n')], {type: 'text/csv'});\n"
    "    var a = document.createElement('a');\n"
    "    a.href = URL.createObjectURL(blob);\n"
    "    a.download = 'domain_report_edited.csv';\n"
    "    a.click();\n"
    "}\n"
    "\n"
    "window.addEventListener('load', function() {\n"
    "    setTimeout(function() {\n"
    "        for (var key in window) {\n"
    "            try {\n"
    "                if (window[key] && window[key].body && window[key].on) {\n"
    "                    window[key].on('click', function(params) {\n"
    "                        if (params.nodes.length > 0) showPanel(params.nodes[0]);\n"
    "                    });\n"
    "                    break;\n"
    "                }\n"
    "            } catch(e) {};\n"
    "        }\n"
    "    }, 1000);\n"
    "});\n"
    "</script>\n"
)

html = html.replace("</style>", "</style>" + side_panel_css)
html = html.replace("</body>", side_panel_html + side_panel_js + "</body>")

with open("domain_network.html", "w", encoding="utf-8") as f_out:
    f_out.write(html)

idisplay(IFrame("domain_network.html", width="100%", height="950px"))
print("[DONE] Saved domain_network.html")
print("TIP: Right-click the file in the Colab sidebar and open in new tab for true full screen")

[DONE] Saved domain_network.html
TIP: Right-click the file in the Colab sidebar and open in new tab for true full screen
